# SVS Annotation Visualization
- **G**: Gland forming cancer (green)
- **S**: Solid cancer / non-gland forming (yellow)

Aperio annotation structure:
- `NegativeROA=0`: Positive region (outer boundary)
- `NegativeROA=1`: Negative region (excluded area inside, drawn with negative pen)

In [ ]:
import openslide
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from lxml import etree
from pathlib import Path

In [ ]:
# Paths
svs_path = '/app/Gland_Seg/S14/SVS/S14-177-1-5.svs'
xml_path = '/app/Gland_Seg/S14/Annotation/S14-177-1-5_S.xml'

# Open slide and read thumbnail
slide = openslide.OpenSlide(svs_path)
print(f'Slide dimensions (level 0): {slide.dimensions}')
print(f'Level dimensions: {slide.level_dimensions}')
print(f'Level downsamples: {slide.level_downsamples}')

viz_level = min(3, slide.level_count - 1)
downsample = slide.level_downsamples[viz_level]
thumb = slide.read_region((0, 0), viz_level, slide.level_dimensions[viz_level]).convert('RGB')
thumb_np = np.array(thumb)
print(f'Using level {viz_level}, downsample={downsample:.1f}, thumbnail size={thumb_np.shape[:2]}')

In [ ]:
def parse_annotations(xml_path):
    """Parse Aperio XML annotation file.
    Returns list of dicts with 'vertices', 'negative' (bool), 'annotation_id', 'region_id'.
    """
    tree = etree.parse(xml_path)
    root = tree.getroot()
    regions_list = []

    for annotation in root.findall('.//Annotation'):
        ann_id = annotation.get('Id')
        line_color = int(annotation.get('LineColor', '0'))

        for region in annotation.findall('.//Region'):
            vertices = []
            for vertex in region.findall('.//Vertex'):
                x = float(vertex.get('X'))
                y = float(vertex.get('Y'))
                vertices.append([x, y])
            if vertices:
                negative = region.get('NegativeROA', '0') == '1'
                regions_list.append({
                    'vertices': np.array(vertices),
                    'annotation_id': ann_id,
                    'region_id': region.get('Id'),
                    'negative': negative,
                    'area': float(region.get('Area', 0)),
                })

    return regions_list

regions = parse_annotations(xml_path)
pos_regions = [r for r in regions if not r['negative']]
neg_regions = [r for r in regions if r['negative']]
print(f'Total: {len(regions)} regions')
print(f'  Positive (NegativeROA=0): {len(pos_regions)} - outer boundary')
print(f'  Negative (NegativeROA=1): {len(neg_regions)} - excluded areas')

In [ ]:
# Annotation type from filename suffix
ann_type = Path(xml_path).stem.rsplit('_', 1)[-1]  # 'S' or 'G'
label_map = {'S': 'Solid cancer', 'G': 'Gland forming cancer'}
pos_color = {'S': (1.0, 1.0, 0.0), 'G': (0.0, 1.0, 0.0)}  # yellow / green
neg_color = (1.0, 0.0, 0.0)  # red for negative regions

display_color = pos_color.get(ann_type, (1.0, 0.5, 0.0))
display_label = label_map.get(ann_type, f'Unknown ({ann_type})')

# --- Full overview: 3 panels ---
fig, axes = plt.subplots(1, 3, figsize=(30, 10))

# Panel 1: Original
axes[0].imshow(thumb_np)
axes[0].set_title('Original Slide', fontsize=14)
axes[0].axis('off')

# Panel 2: Positive only (outer boundary)
axes[1].imshow(thumb_np)
patches_pos = []
for r in pos_regions:
    poly = Polygon(r['vertices'] / downsample, closed=True)
    patches_pos.append(poly)
pc_pos = PatchCollection(patches_pos, facecolor=(*display_color, 0.25), edgecolor=display_color, linewidth=2.0)
axes[1].add_collection(pc_pos)
axes[1].set_title(f'Positive region (outer boundary)\n{display_label}', fontsize=14)
axes[1].axis('off')

# Panel 3: Positive + Negative overlay
axes[2].imshow(thumb_np)
# Draw positive first
patches_pos2 = []
for r in pos_regions:
    poly = Polygon(r['vertices'] / downsample, closed=True)
    patches_pos2.append(poly)
pc_pos2 = PatchCollection(patches_pos2, facecolor=(*display_color, 0.25), edgecolor=display_color, linewidth=2.0)
axes[2].add_collection(pc_pos2)
# Draw negative on top
patches_neg = []
for r in neg_regions:
    poly = Polygon(r['vertices'] / downsample, closed=True)
    patches_neg.append(poly)
pc_neg = PatchCollection(patches_neg, facecolor=(*neg_color, 0.3), edgecolor=neg_color, linewidth=1.5)
axes[2].add_collection(pc_neg)
axes[2].set_title(f'Positive + Negative (excluded)\n{len(pos_regions)} positive, {len(neg_regions)} negative', fontsize=14)
axes[2].axis('off')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=(*display_color, 0.25), edgecolor=display_color, linewidth=2, label=f'Positive: {display_label}'),
    Patch(facecolor=(*neg_color, 0.3), edgecolor=neg_color, linewidth=2, label='Negative: excluded area'),
]
axes[2].legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('/app/Gland_Seg/Code/annotation_viz.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Zoomed view around annotation bounding box ---
all_verts = np.concatenate([r['vertices'] for r in regions], axis=0)
x_min, y_min = all_verts.min(axis=0)
x_max, y_max = all_verts.max(axis=0)
print(f'Annotation bounding box (level 0): x=[{x_min:.0f}, {x_max:.0f}], y=[{y_min:.0f}, {y_max:.0f}]')

# Add 10% padding
pad_x = (x_max - x_min) * 0.1
pad_y = (y_max - y_min) * 0.1
crop_x = int(max(0, x_min - pad_x))
crop_y = int(max(0, y_min - pad_y))
crop_w = int(min(slide.dimensions[0], x_max + pad_x) - crop_x)
crop_h = int(min(slide.dimensions[1], y_max + pad_y) - crop_y)

crop_level = min(2, slide.level_count - 1)
crop_ds = slide.level_downsamples[crop_level]
crop_size = (int(crop_w / crop_ds), int(crop_h / crop_ds))
crop_img = slide.read_region((crop_x, crop_y), crop_level, crop_size).convert('RGB')
crop_np = np.array(crop_img)
print(f'Crop at level {crop_level} (ds={crop_ds:.1f}): {crop_np.shape}')

fig, ax = plt.subplots(figsize=(18, 14))
ax.imshow(crop_np)

# Positive regions
patches_pos = []
for r in pos_regions:
    shifted = (r['vertices'] - np.array([crop_x, crop_y])) / crop_ds
    patches_pos.append(Polygon(shifted, closed=True))
pc_pos = PatchCollection(patches_pos, facecolor=(*display_color, 0.15), edgecolor=display_color, linewidth=2.5)
ax.add_collection(pc_pos)

# Negative regions
patches_neg = []
for r in neg_regions:
    shifted = (r['vertices'] - np.array([crop_x, crop_y])) / crop_ds
    patches_neg.append(Polygon(shifted, closed=True))
pc_neg = PatchCollection(patches_neg, facecolor=(*neg_color, 0.25), edgecolor=neg_color, linewidth=1.5)
ax.add_collection(pc_neg)

ax.set_title(f'Zoomed: {display_label} ({ann_type})\n'
             f'{len(pos_regions)} positive (yellow outline) / {len(neg_regions)} negative (red, excluded)', fontsize=14)
ax.legend(handles=legend_elements, loc='lower right', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig('/app/Gland_Seg/Code/annotation_viz_zoomed.png', dpi=150, bbox_inches='tight')
plt.show()